In [10]:
from pathlib import Path
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

print("Environment ready.")

Environment ready.


In [11]:
PROJECT_DIR = Path(r"C:\Users\bimal\OneDrive\Desktop\EVIDENT")

OPENPHISH_PATH = PROJECT_DIR / "data" / "raw" / "phishing" / "model 1" / "OpenPish.txt"
PHIUSIIL_PATH = PROJECT_DIR / "data" / "raw" / "phishing" / "model 1" / "PhiUSIIL_Phishing_URL_Dataset.csv"

MODEL_PATH = PROJECT_DIR / "models" / "phishing" / "phishing_model.pkl"
METADATA_PATH = PROJECT_DIR / "models" / "phishing" / "model_metadata.json"

REPORT_DIR = PROJECT_DIR / "reports" / "phishing"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in [OPENPHISH_PATH, PHIUSIIL_PATH, MODEL_PATH, METADATA_PATH]:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}")

OK  C:\Users\bimal\OneDrive\Desktop\EVIDENT\data\raw\phishing\model 1\OpenPish.txt
OK  C:\Users\bimal\OneDrive\Desktop\EVIDENT\data\raw\phishing\model 1\PhiUSIIL_Phishing_URL_Dataset.csv
OK  C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\phishing\phishing_model.pkl
OK  C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\phishing\model_metadata.json


In [3]:
model = joblib.load(MODEL_PATH)
print("Loaded model:", type(model).__name__)


Loaded model: Pipeline


In [12]:
model = joblib.load(MODEL_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

MODEL_FEATURES = metadata["features"]

print("Model:", metadata["model_name"])
print("Task:", metadata["task"])
print("Expected feature count:", metadata["feature_count"])
print("Loaded feature count:", len(MODEL_FEATURES))
print("Feature schema valid:", len(MODEL_FEATURES) == metadata["feature_count"])

Model: Random Forest
Task: binary phishing URL classification
Expected feature count: 50
Loaded feature count: 50
Feature schema valid: True


In [13]:
def load_openphish(path: Path) -> pd.DataFrame:
    """
    Read OpenPhish as a text feed without contacting any URL.

    Supports:
    - one URL per line
    - whitespace-delimited URL records
    - simple CSV-like lines where the first URL is used
    """
    raw = path.read_text(encoding="utf-8", errors="ignore")

    rows = []
    for line in raw.splitlines():
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        # Extract the first HTTP(S) URL from the line.
        match = re.search(r"https?://[^\s,;]+", line, flags=re.IGNORECASE)
        if match:
            rows.append(match.group(0).strip())

    out = pd.DataFrame({"URL": rows})
    out["URL"] = out["URL"].astype(str).str.strip()
    out = out[out["URL"].str.len() > 0].copy()

    return out.reset_index(drop=True)


openphish = load_openphish(OPENPHISH_PATH)

print("OpenPhish URLs loaded:", len(openphish))
display(openphish.head(10))

OpenPhish URLs loaded: 300


,URL
0,http://www.supershopf.vip/
1,https://facebook-auto-liker.blogspot.com/
2,https://nhengenharia.com/crchildrenscare/crchi...
3,https://facebook-logiin.vercel.app/
4,https://roblox.com.bo/users/3950918659/profile
5,http://trezweb3.cam/
6,https://tokenim-cdn-sg-k.lol/
7,https://mtoken-cdn-hk.lol/
8,https://webmail.50-6-22-93.nip.io/desktop/?aut...
9,https://qr2.it/Go/2838281


In [8]:
WEBPAGE_FEATURES = [
    "URLSimilarityIndex",
    "CharContinuationRate",
    "TLDLegitimateProb",
    "URLCharProb",
    "LineOfCode",
    "LargestLineLength",
    "HasTitle",
    "DomainTitleMatchScore",
    "URLTitleMatchScore",
    "HasFavicon",
    "Robots",
    "IsResponsive",
    "NoOfURLRedirect",
    "NoOfSelfRedirect",
    "HasDescription",
    "NoOfPopup",
    "NoOfiFrame",
    "HasExternalFormSubmit",
    "HasSocialNet",
    "HasSubmitButton",
    "HasHiddenFields",
    "HasPasswordField",
    "Bank",
    "Pay",
    "Crypto",
    "HasCopyrightInfo",
    "NoOfImage",
    "NoOfCSS",
    "NoOfJS",
    "NoOfSelfRef",
    "NoOfEmptyRef",
    "NoOfExternalRef",
]

print("Webpage-dependent feature count:", len(WEBPAGE_FEATURES))


Webpage-dependent feature count: 32


In [16]:
from urllib.parse import urlsplit, urlunsplit

def normalize_url(url: str) -> str:
    url = str(url).strip()

    if not url:
        return ""

    # Remove surrounding quotes.
    url = url.strip("")

    # Normalize scheme and host casing while preserving path/query.
    try:
        parsed = urlsplit(url)

        if not parsed.scheme or not parsed.netloc:
            return url.lower().rstrip("/")

        host = parsed.hostname.lower() if parsed.hostname else ""
        port = ""

        if parsed.port:
            # Keep non-default ports.
            if not (
                (parsed.scheme.lower() == "http" and parsed.port == 80)
                or (parsed.scheme.lower() == "https" and parsed.port == 443)
            ):
                port = f":{parsed.port}"

        netloc = host + port

        return urlunsplit(
            (
                parsed.scheme.lower(),
                netloc,
                parsed.path or "",
                parsed.query or "",
                parsed.fragment or "",
            )
        ).rstrip("/")
    except Exception:
        return url.lower().rstrip("/")


openphish["normalized_url"] = openphish["URL"].map(normalize_url)

before = len(openphish)
openphish = openphish.drop_duplicates("normalized_url").reset_index(drop=True)

print("Original URL records :", before)
print("Unique normalized URLs:", len(openphish))
print("Duplicates removed    :", before - len(openphish))

Original URL records : 300
Unique normalized URLs: 298
Duplicates removed    : 2


In [17]:
# Load only the columns needed for the reference lookup.
phi_columns = ["URL"] + MODEL_FEATURES + ["label"]

phi = pd.read_csv(
    PHIUSIIL_PATH,
    usecols=phi_columns,
    low_memory=False,
)

print("PhiUSIIL rows:", len(phi))
print("PhiUSIIL columns loaded:", len(phi.columns))

missing_features = [c for c in MODEL_FEATURES if c not in phi.columns]

if missing_features:
    raise ValueError(f"Missing model features in PhiUSIIL: {missing_features}")

phi["normalized_url"] = phi["URL"].map(normalize_url)

# Keep one reference row per normalized URL.
phi_reference = (
    phi.drop_duplicates("normalized_url")
       .reset_index(drop=True)
)

print("Unique normalized PhiUSIIL URLs:", len(phi_reference))

PhiUSIIL rows: 235795
PhiUSIIL columns loaded: 52
Unique normalized PhiUSIIL URLs: 234188


In [18]:
matched = openphish.merge(
    phi_reference[["normalized_url", "URL"] + MODEL_FEATURES + ["label"]],
    on="normalized_url",
    how="left",
    suffixes=("_openphish", "_phius iil"),
)

# The model features themselves are not duplicated, so they remain unchanged.
feature_available = matched[MODEL_FEATURES].notna().all(axis=1)

matched_count = int(feature_available.sum())
unmatched_count = len(matched) - matched_count

print("OpenPhish URLs:", len(matched))
print("URLs with all 50 reference features:", matched_count)
print("URLs without reference features:", unmatched_count)
print("Reference coverage: {:.2f}%".format(
    100 * matched_count / max(len(matched), 1)
))

OpenPhish URLs: 298
URLs with all 50 reference features: 0
URLs without reference features: 298
Reference coverage: 0.00%


In [19]:
reference_rows = matched.loc[feature_available].copy()

if reference_rows.empty:
    print("No OpenPhish URLs overlap with PhiUSIIL.")
else:
    X_external_reference = reference_rows[MODEL_FEATURES].copy()

    # OpenPhish entries are treated as phishing.
    y_external_reference = np.ones(len(reference_rows), dtype=int)

    predictions = model.predict(X_external_reference)
    predictions = np.asarray(predictions).astype(int)

    reference_rows["prediction"] = predictions
    reference_rows["expected_label"] = y_external_reference
    reference_rows["correct"] = (
        reference_rows["prediction"] == reference_rows["expected_label"]
    )

    print("Reference validation results")
    print("-" * 40)
    print("Samples:", len(reference_rows))
    print("Accuracy :", accuracy_score(y_external_reference, predictions))
    print("Precision:", precision_score(y_external_reference, predictions, zero_division=0))
    print("Recall   :", recall_score(y_external_reference, predictions, zero_division=0))
    print("F1-score :", f1_score(y_external_reference, predictions, zero_division=0))
    print()
    print("Confusion matrix:")
    print(confusion_matrix(y_external_reference, predictions))

    display(
        reference_rows[
            ["URL_openphish", "prediction", "expected_label", "correct"]
        ].head(20)
    )

No OpenPhish URLs overlap with PhiUSIIL.


In [20]:
coverage_report = pd.DataFrame({
    "metric": [
        "openphish_total_urls",
        "openphish_unique_urls",
        "reference_feature_matches",
        "reference_feature_coverage_percent",
    ],
    "value": [
        len(openphish),
        len(openphish),
        matched_count,
        round(100 * matched_count / max(len(openphish), 1), 4),
    ],
})

coverage_path = REPORT_DIR / "openphish_reference_coverage.csv"
coverage_report.to_csv(coverage_path, index=False)

if not reference_rows.empty:
    prediction_path = REPORT_DIR / "openphish_reference_predictions.csv"
    reference_rows.to_csv(prediction_path, index=False)
    print("Saved:", prediction_path)

print("Saved:", coverage_path)

Saved: C:\Users\bimal\OneDrive\Desktop\EVIDENT\reports\phishing\openphish_reference_coverage.csv


In [21]:
# Final schema check for the production inference contract.
print("Production model contract")
print("=" * 40)
print("Expected features:", len(MODEL_FEATURES))
print("Model object:", type(model).__name__)

missing_from_reference = [
    feature for feature in MODEL_FEATURES
    if feature not in phi.columns
]

if missing_from_reference:
    print("Missing:", missing_from_reference)
else:
    print("All 50 model features are available in PhiUSIIL.")

print("\nModel expects a feature matrix, not raw URL strings.")

Production model contract
Expected features: 50
Model object: Pipeline
All 50 model features are available in PhiUSIIL.

Model expects a feature matrix, not raw URL strings.
